## Extracting the complexity measures 

In [1]:
import os, random, sys, tqdm, torch, warnings

In [2]:
import numpy as np
import pandas as pd

In [3]:
from itertools import islice
from multiprocessing import Pool
from pymfe.mfe import MFE

In [4]:
sys.path.append('../src')
from tabicl_prior import sample_datasets

In [5]:
warnings.filterwarnings('ignore', category=RuntimeWarning)

In [6]:
SEED  = 42

In [7]:
mode = os.environ['MODE']
jobs = int(os.environ['JOBS'])

if mode == 'synthetic':
    stage = int(os.environ['STAGE'])
    n_datasets = int(os.environ['DATASETS'])
    prior = int(os.environ['PRIOR'])

In [8]:
random.seed(SEED)
np.random.seed(SEED)

## Helper Functions

In [9]:
def normalize(file):
    df = pd.read_csv(file, na_values=['?', 'Not_Say'], low_memory=False)
    X, y = df.iloc[:, :-1], df.iloc[:, -1]

    X = X.dropna(axis=1, how='all')
    cat = X.select_dtypes(exclude='number').columns
    num = X.columns.difference(cat)

    X[num] = X[num].fillna(X[num].mean())
    X[cat] = X[cat].fillna('missing')

    return X.to_numpy(), y.to_numpy()

In [10]:
def load_datasets(path):
    return [(file, *normalize(f'{path}/{file}')) for file in os.listdir(path)]

In [11]:
def extract(args):
    i, X, y = args
    mfe = MFE(groups='complexity', features='all', summary='mean', random_state=SEED)
    mfe.fit(X, y, suppress_warnings=True)
    measures, values = mfe.extract(suppress_warnings=True)
    return {'id': i, 'n_classes': len(np.unique(y)), 'n_rows': X.shape[0], 'n_cols': X.shape[1], **dict(zip(measures, map(float, values)))}

## Real datasets

In [12]:
if mode == 'real':
    data_tuples = load_datasets('../data/processed/tabarena/') + load_datasets('../data/processed/talent/')
    with Pool(jobs) as pool:
        complexity = list(tqdm.tqdm(pool.imap(extract, data_tuples)))
    df = pd.DataFrame(complexity)
    df.to_csv('real.csv')

## Synthetic datasets

In [13]:
if mode == 'synthetic':
    synthetic = sample_datasets('clf', stage=stage, n_datasets=n_datasets, seed=SEED, n_jobs=prior)
    data_tuples = ((i, d['X'], d['y']) for i, d in enumerate(synthetic))
    with Pool(jobs) as pool:
        complexity = list(tqdm.tqdm(pool.imap(extract, data_tuples)))
    pd.DataFrame(complexity).to_csv(f'../results/synthetic_stage{stage}.csv', index=False)

0it [00:00, ?it/s]

1it [00:34, 34.18s/it]

3it [00:35,  9.28s/it]

4it [00:38,  7.09s/it]

5it [00:51,  9.27s/it]

7it [00:54,  5.38s/it]

8it [01:21, 10.88s/it]

8it [01:21, 10.15s/it]